# bn-dub — Bengali dubbing console (Colab)

Run the cells top to bottom. The last one prints a public `*.gradio.live` link to the console.

**Keys:** put `ELEVENLABS_API_KEY` and `ANTHROPIC_API_KEY` in Colab's **Secrets** panel (🔑 icon in the left sidebar) and toggle *Notebook access* on. The notebook reads them from there — never paste keys into a cell.

**Runtime:** CPU is fine. Pick a GPU runtime only if you plan to use the `demucs` background mode.

In [ ]:
#@title 1 · Install
REPO = "https://github.com/ayanisone/11.git"  #@param {type:"string"}
BRANCH = "claude/eager-keller-kynj48"  #@param {type:"string"}

import os, subprocess, sys
if not os.path.isdir("/content/bn-dub"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, "/content/bn-dub"], check=True)
%cd /content/bn-dub
!pip install -q -e . 2>&1 | tail -1
!ffmpeg -version | head -1
print("ok")

In [ ]:
#@title 2 · Keys (from Colab Secrets)
import os
from google.colab import userdata

for name in ("ELEVENLABS_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: loaded")
    except Exception as e:  # SecretNotFoundError / NotebookAccessError
        print(f"{name}: MISSING — add it in the Secrets panel and enable notebook access ({type(e).__name__})")

In [ ]:
#@title 3 · (optional) demucs for a cleaner background stem — needs a GPU runtime
INSTALL_DEMUCS = False  #@param {type:"boolean"}
if INSTALL_DEMUCS:
    !pip install -q demucs 2>&1 | tail -1

In [ ]:
#@title 4 · Launch the console
%cd /content/bn-dub
import os, importlib, app
importlib.reload(app)
if not (os.environ.get('ELEVENLABS_API_KEY') and os.environ.get('ANTHROPIC_API_KEY')):
    print('Keys not loaded — run cell 2, or paste them in the "API keys" panel at the top of the console.')
# share=True gives a public gradio.live URL (Colab can't expose localhost). Link is valid for 72 h.
app.launch(app.build(), share=True, debug=True)

## Where the files go

Projects land in `/content/bn-dub/projects/<name>/`. Download `out/dubbed.mp4` or `out/dub.wav` from the Mix tab, or from the Files panel on the left. **Colab wipes `/content` when the runtime ends** — mount Drive and change `BN_DUB_PROJECTS_DIR` if you want projects to persist:

```python
from google.colab import drive; drive.mount('/content/drive')
import os; os.environ['BN_DUB_PROJECTS_DIR'] = '/content/drive/MyDrive/bn-dub-projects'
```

Run that before cell 4.